# Voxelized Event Visualization

Interactive 3D visualization of voxelized photon data from PhotonSim.
Each label is shown in a different color.

In [ ]:
import sys
import os
import numpy as np
import uproot
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add paths for imports
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), '..'))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), '..', '..'))

from voxelize import (
    VoxelGridConfig,
    voxelize_from_photon_indices,
    flat_index_to_position,
    get_voxel_statistics
)

## Configuration

In [ ]:
# Path to ROOT file with multi-particle events (config 009: mu + pi+ + pi-)
ROOT_FILE = "/sdf/data/neutrino/cjesus/dataprod_output_v0/timing_tests_v3/roma/water/uniform_energy/config_000009/output_job_000001.root"

# Color palette for labels (up to 12 distinct colors)
COLORS = [
    'rgb(31, 119, 180)',   # blue
    'rgb(255, 127, 14)',   # orange
    'rgb(44, 160, 44)',    # green
    'rgb(214, 39, 40)',    # red
    'rgb(148, 103, 189)',  # purple
    'rgb(140, 86, 75)',    # brown
    'rgb(227, 119, 194)',  # pink
    'rgb(127, 127, 127)',  # gray
    'rgb(188, 189, 34)',   # olive
    'rgb(23, 190, 207)',   # cyan
    'rgb(255, 187, 120)',  # light orange
    'rgb(152, 223, 138)',  # light green
]

# PDG code to particle name mapping
PDG_NAMES = {
    11: 'e-', -11: 'e+',
    13: 'mu-', -13: 'mu+',
    22: 'gamma',
    111: 'pi0',
    211: 'pi+', -211: 'pi-',
    321: 'K+', -321: 'K-',
    2212: 'proton', -2212: 'antiproton',
}

# Category names
CATEGORY_NAMES = {
    0: 'Primary',
    1: 'DecayElectron',
    2: 'SecondaryPion',
    3: 'Gamma',
    -1: 'Unknown'
}

def get_particle_name(pdg):
    return PDG_NAMES.get(int(pdg), f'PDG_{pdg}')

def get_category_name(cat):
    return CATEGORY_NAMES.get(int(cat), f'Cat_{cat}')

## Load ROOT File

In [ ]:
# Open ROOT file and get number of events
root_file = uproot.open(ROOT_FILE)
tree = root_file['OpticalPhotons']
n_events = tree.num_entries

print(f"ROOT file: {ROOT_FILE}")
print(f"Number of events: {n_events}")

In [ ]:
def load_event_data(entry_index):
    """Load photon data and label info for a single event."""
    branches = [
        'PhotonPosX', 'PhotonPosY', 'PhotonPosZ',
        'NLabels', 'Label_PhotonIDsSize', 'Label_PhotonIDsData',
        'PrimaryEnergy', 'NOpticalPhotons',
        'TrackInfo_PDG', 'TrackInfo_Category', 'TrackInfo_Energy'
    ]
    
    data = tree.arrays(branches, entry_start=entry_index, entry_stop=entry_index+1, library='np')
    
    # Convert positions from mm to meters
    pos_x = data['PhotonPosX'][0] / 1000.0
    pos_y = data['PhotonPosY'][0] / 1000.0
    pos_z = data['PhotonPosZ'][0] / 1000.0
    positions = np.column_stack((pos_x, pos_y, pos_z))
    
    # Parse label information
    n_labels = int(data['NLabels'][0])
    photon_ids_sizes = data['Label_PhotonIDsSize'][0]
    photon_ids_data = data['Label_PhotonIDsData'][0]
    
    # Build list of photon indices per label
    label_photon_indices = []
    offset = 0
    for size in photon_ids_sizes:
        size = int(size)
        indices = np.array(photon_ids_data[offset:offset+size], dtype=np.int64)
        label_photon_indices.append(indices)
        offset += size
    
    # Track info (limited to n_labels)
    track_pdgs = data['TrackInfo_PDG'][0][:n_labels]
    track_categories = data['TrackInfo_Category'][0][:n_labels]
    track_energies = data['TrackInfo_Energy'][0][:n_labels]
    
    return {
        'positions': positions,
        'n_labels': n_labels,
        'label_photon_indices': label_photon_indices,
        'primary_energy': float(data['PrimaryEnergy'][0]),
        'n_photons': int(data['NOpticalPhotons'][0]),
        'track_pdgs': track_pdgs,
        'track_categories': track_categories,
        'track_energies': track_energies
    }

## Voxelization and Visualization Functions

In [ ]:
def create_voxel_visualization(event_idx, size_scale=1.0, opacity=0.8, show_all=True):
    """Create 3D visualization of voxelized event."""
    
    # Load event data
    event_data = load_event_data(event_idx)
    config = VoxelGridConfig()
    
    # Voxelize
    voxel_data = voxelize_from_photon_indices(
        event_data['positions'],
        event_data['label_photon_indices'],
        config
    )
    
    stats = get_voxel_statistics(voxel_data)
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each label
    for label_idx in range(voxel_data['n_labels']):
        flat_indices = voxel_data['voxel_indices'][label_idx]
        counts = voxel_data['voxel_counts'][label_idx]
        
        if len(flat_indices) == 0:
            continue
        
        # Get voxel center positions
        positions = flat_index_to_position(flat_indices, config, center=True)
        
        # Scale marker size by photon count (log scale for better visibility)
        sizes = np.log10(counts + 1) * 5 * size_scale + 2
        
        # Get label info
        pdg = event_data['track_pdgs'][label_idx]
        category = event_data['track_categories'][label_idx]
        energy = event_data['track_energies'][label_idx]
        n_photons = np.sum(counts)
        
        particle_name = get_particle_name(pdg)
        category_name = get_category_name(category)
        
        label_text = f"{particle_name} ({category_name})"
        
        # Hover text
        hover_text = [
            f"Label {label_idx}: {label_text}<br>"
            f"Position: ({positions[i,0]:.3f}, {positions[i,1]:.3f}, {positions[i,2]:.3f}) m<br>"
            f"Photons in voxel: {counts[i]}<br>"
            f"Energy: {energy:.1f} MeV"
            for i in range(len(positions))
        ]
        
        color = COLORS[label_idx % len(COLORS)]
        
        fig.add_trace(go.Scatter3d(
            x=positions[:, 0],
            y=positions[:, 1],
            z=positions[:, 2],
            mode='markers',
            marker=dict(
                size=sizes,
                color=color,
                opacity=opacity,
                line=dict(width=0)
            ),
            name=f"L{label_idx}: {label_text} ({n_photons:,} ph, {len(flat_indices)} vox)",
            text=hover_text,
            hoverinfo='text',
            visible=True if show_all else (label_idx == 0)
        ))
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f"Event {event_idx} | {event_data['n_photons']:,} photons | "
                 f"{stats['total_nonzero_voxels']:,} voxels | "
                 f"{event_data['n_labels']} labels | "
                 f"{event_data['primary_energy']:.1f} MeV",
            x=0.5
        ),
        scene=dict(
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            aspectmode='data'
        ),
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
            bgcolor="rgba(255,255,255,0.8)"
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=700
    )
    
    return fig, event_data, stats

## Interactive Visualization

In [ ]:
# Create widgets
event_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=n_events - 1,
    step=1,
    description='Event:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

size_slider = widgets.FloatSlider(
    value=1.0,
    min=0.2,
    max=3.0,
    step=0.1,
    description='Marker size:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

opacity_slider = widgets.FloatSlider(
    value=0.8,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Opacity:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

output = widgets.Output()

def update_plot(change):
    with output:
        clear_output(wait=True)
        fig, event_data, stats = create_voxel_visualization(
            event_slider.value,
            size_scale=size_slider.value,
            opacity=opacity_slider.value
        )
        fig.show()
        
        # Print event summary
        print(f"\nEvent {event_slider.value} Summary:")
        print(f"  Primary energy: {event_data['primary_energy']:.1f} MeV")
        print(f"  Total photons: {event_data['n_photons']:,}")
        print(f"  Total voxels: {stats['total_nonzero_voxels']:,}")
        print(f"  Compression: {stats['total_photons'] / max(stats['total_nonzero_voxels'], 1):.1f}x")
        print(f"\n  Labels ({event_data['n_labels']}):")
        for i in range(event_data['n_labels']):
            pdg = event_data['track_pdgs'][i]
            cat = event_data['track_categories'][i]
            energy = event_data['track_energies'][i]
            n_ph = stats['photons_per_label'][i]
            print(f"    {i}: {get_particle_name(pdg):6s} ({get_category_name(cat):14s}) - {energy:8.1f} MeV, {n_ph:8,} photons")

# Connect widgets to update function
event_slider.observe(update_plot, names='value')
size_slider.observe(update_plot, names='value')
opacity_slider.observe(update_plot, names='value')

# Layout
controls = widgets.VBox([
    widgets.HBox([event_slider, size_slider, opacity_slider])
])

display(controls, output)

# Initial plot
update_plot(None)

## Quick Event Navigation

Use this cell to quickly jump to specific events:

In [ ]:
# Change event_idx to view different events
event_idx = 1

fig, event_data, stats = create_voxel_visualization(event_idx, size_scale=1.0, opacity=0.8)
fig.show()

print(f"\nEvent {event_idx} Summary:")
print(f"  Primary energy: {event_data['primary_energy']:.1f} MeV")
print(f"  Total photons: {event_data['n_photons']:,}")
print(f"  Total voxels: {stats['total_nonzero_voxels']:,}")
print(f"  Compression: {stats['total_photons'] / max(stats['total_nonzero_voxels'], 1):.1f}x")
print(f"\n  Labels ({event_data['n_labels']}):")
for i in range(event_data['n_labels']):
    pdg = event_data['track_pdgs'][i]
    cat = event_data['track_categories'][i]
    energy = event_data['track_energies'][i]
    n_ph = stats['photons_per_label'][i]
    print(f"    {i}: {get_particle_name(pdg):6s} ({get_category_name(cat):14s}) - {energy:8.1f} MeV, {n_ph:8,} photons")

## Compare Multiple Events Side-by-Side

In [ ]:
def create_multi_event_comparison(event_indices, size_scale=0.8, opacity=0.7):
    """Create side-by-side comparison of multiple events."""
    n_events_to_show = len(event_indices)
    
    fig = make_subplots(
        rows=1, cols=n_events_to_show,
        specs=[[{'type': 'scatter3d'} for _ in range(n_events_to_show)]],
        subplot_titles=[f"Event {idx}" for idx in event_indices],
        horizontal_spacing=0.02
    )
    
    config = VoxelGridConfig()
    
    for col_idx, event_idx in enumerate(event_indices):
        event_data = load_event_data(event_idx)
        voxel_data = voxelize_from_photon_indices(
            event_data['positions'],
            event_data['label_photon_indices'],
            config
        )
        
        for label_idx in range(voxel_data['n_labels']):
            flat_indices = voxel_data['voxel_indices'][label_idx]
            counts = voxel_data['voxel_counts'][label_idx]
            
            if len(flat_indices) == 0:
                continue
            
            positions = flat_index_to_position(flat_indices, config, center=True)
            sizes = np.log10(counts + 1) * 5 * size_scale + 2
            
            pdg = event_data['track_pdgs'][label_idx]
            particle_name = get_particle_name(pdg)
            color = COLORS[label_idx % len(COLORS)]
            
            fig.add_trace(
                go.Scatter3d(
                    x=positions[:, 0],
                    y=positions[:, 1],
                    z=positions[:, 2],
                    mode='markers',
                    marker=dict(size=sizes, color=color, opacity=opacity),
                    name=f"E{event_idx} L{label_idx}: {particle_name}",
                    showlegend=(col_idx == 0)
                ),
                row=1, col=col_idx + 1
            )
    
    fig.update_layout(
        title="Multi-Event Comparison",
        height=600,
        margin=dict(l=0, r=0, t=60, b=0)
    )
    
    return fig

# Compare events 0, 1, 2
fig = create_multi_event_comparison([0, 1, 2])
fig.show()